# DFND on the wide, sparsely sampled tube

Can the native method carry the independently established geometry all the way
into `Topography`? At 1.4 Å, it resolves the two hull openings and the channel's
two physical contacts. At 2.5 Å, a clear native candidate exists but the boundary
calculation reports a numerical ambiguity. The 5 Å exclusion control exposes
a separate question about local candidates inside a covered all-atom carrier.

Read [the system notebook](system.ipynb) for the independent proofs. This notebook
runs independently from its case directory. It executes the actual public DFND
route, keeps unresolved outcomes visible, and makes no original-provider or
server call. Run All does not overwrite retained evidence.

In [ ]:
from pathlib import Path
from html import escape
import hashlib
import json
import sys
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
import pyunitwizard as puw
import study

CASE = Path.cwd()
ARTIFACTS = CASE / 'artifacts/straight_tube_wide_sparse_v1'
fixture = study.read(ARTIFACTS / 'input.json')
protocol = study.read(ARTIFACTS / 'protocol.json')
assert hashlib.sha256((ARTIFACTS / 'input.json').read_bytes()).hexdigest() == protocol['input_sha256']
centers = np.asarray(puw.get_value(study.quantity(fixture['coordinates'], 'coordinates'), to_unit='angstroms'))
radii = np.asarray(puw.get_value(study.quantity(fixture['atom_radii'], 'atom_radii'), to_unit='angstroms'))
assert centers.shape == (77, 3) and np.allclose(radii, 1.88, rtol=0, atol=1e-14)
assert fixture['source_atom_indices'] == list(range(77))

def table(headers, rows):
    def cells(values, tag):
        return ''.join(f'<{tag}>{escape(str(value))}</{tag}>' for value in values)
    display(HTML('<table><thead><tr>' + cells(headers, 'th') +
                 '</tr></thead><tbody>' + ''.join('<tr>' + cells(row, 'td') +
                 '</tr>' for row in rows) + '</tbody></table>'))

## Run the fixed model through the public API

The helper restores full-precision coordinates over the PDB topology, verifies
MolSysMT's 1.88 Å radii and checks source atom IDs. All three calls use
`get_topography(method='dfnd')`, zero physical residence/permeability tolerances,
`with_connectors`, `flag_only`, `dryface`, epsilon and boundary margin 1e-6 Å,
and `min_size=0`. The explicit query converts Å into the native nm convention.
The raw/typed/public audit checks complete wet/dry node inventories, source
supports, parent/child relations and recoverable physical evidence.

In [ ]:
import topomt
import ackredit

reader_protocol = study.read(ARTIFACTS / 'reader_protocol.json')
print('Imported TopoMT:', topomt.__file__)
print('Producer version:', topomt.__version__)
print('Reviewed source revision:', reader_protocol['topomt_source_revision'])
references = study.references(CASE)
results, observations, candidates = {}, {}, {}
with ackredit.capture('Wide tube public notebook') as credit:
    for probe in (1.4, 2.5, 5.0):
        topo = study.receive(CASE, probe)
        results[probe] = topo
        observations[probe] = study.describe(topo)
        candidates[probe] = study.inspect_centerlines(topo)
current_credit = credit.attribution.to_dict()
assert current_credit
original = study.read(ARTIFACTS / 'native_observations.json')
assert original['input_sha256'] == protocol['input_sha256']

## Graph objects and physical openings answer different questions

There are 347 Delaunay tetrahedra in this input. A wet graph component is assembled
from local residence/transit rules. Public catalog objects and Mouth children
retain those definitions; they do not automatically count all-atom free regions
or hull openings. The table reports those quantities separately. Unknown
physical counts remain unknown, never zero.

In [ ]:
rows = []
for probe, topo in results.items():
    observed = observations[probe]
    physical = topo.dfnd.raw['boundary_connectivity']
    saved = next(r for r in original['probes'] if r['probe_angstroms'] == probe)
    assert observed['node_inventory'] == saved['public_observation']['node_inventory']
    assert [f['python_class'] for f in observed['public_features']] == [
        f['python_class'] for f in saved['public_observation']['public_features']]
    assert physical['status'] == saved['raw']['boundary_connectivity']['status']
    assert physical['n_patches'] == saved['raw']['boundary_connectivity']['n_patches']
    features = observed['public_features']
    mouths = sum(f['python_class'] == 'Mouth' for f in features)
    rows.append([probe, observed['node_inventory']['wet_nodes'],
        len(observed['wet_components']), len(topo.dfnd.raw['external_links']),
        len(features) - mouths, mouths, physical['status'],
        physical['n_patches'] if physical['n_patches'] is not None else 'unknown'])
table(['Probe (Å)', 'Wet nodes', 'Wet components', 'Native links',
       'Other public objects', 'Mouth children', 'Physical status', 'Physical patches'], rows)

At **1.4 Å**, the complete hull has two physical patches, matching the independent
reference. The Channel has two path-supported contacts. Other catalog objects
are additional **local candidates**: their ten attempted component/patch
associations remain unresolved. They do not demonstrate extra side-wall holes;
the independent wall proof excludes those. The global association status can
therefore remain unresolved while this Channel's two contacts are resolved.

At **2.5 Å**, the boundary reason is `face 12: critical-section ordering within
numerical margin`. This is a truthful unavailable result, not a computed patch
count contradicting two. [#184](https://github.com/uibcdf/topomt/issues/184) owns
its diagnosis. The nine native wet components yield eight non-Mouth public objects: catalog
promotion need not produce one object per graph component; the native bank
retains the full inventory. A public `Percolating` object is a native graph classification,
not an independently certified additional physical passage.

At **5 Å**, the reason is `face 10: arrangement work budget exhausted`, tracked
under [#180](https://github.com/uibcdf/topomt/issues/180). The surviving
OpenConcavity/Mouth pair must be interpreted against the independent empty
all-atom carrier. Local three/four-atom operators do not certify exclusion by
all atoms. The support diagnosis and public qualification policy remain
[#181](https://github.com/uibcdf/topomt/issues/181), `needs_review`; this result
alone does not diagnose an incorrect local solver.

In [ ]:
rows = []
for probe, observed in observations.items():
    for feature in observed['public_features']:
        if feature['python_class'] == 'Mouth':
            continue
        physical = feature.get('physical_openings') or {}
        rows.append([probe, feature['feature_id'], feature['python_class'],
            ', '.join(feature['child_ids']) or 'none',
            len(feature['lining_source_atom_indices']),
            physical.get('status', 'unavailable'),
            physical.get('n_patches') if physical.get('n_patches') is not None else 'unknown'])
table(['Probe (Å)', 'Public ID', 'Class', 'Child IDs', 'Lining atoms',
       'Physical contacts', 'Contact count'], rows)

The complete detached records retain component keys, support keys, original source
atom IDs and the underlying tetrahedron/face geometry. A carrier volume or local
solvent proxy is not the physical volume of the free set. This unit does not use
those metrics to refute the exclusion proof or claim volume accuracy.

## Are the native candidate paths clear?

For native components with multiple exterior links we inspect `channel_skeleton`
and independently test every segment against every input atom. The method's
`is_collision_validated` flag remains **False**; reading or checking a candidate
does not rewrite its original qualification.

In [ ]:
rows = []
for probe, paths in candidates.items():
    for path in paths:
        assert path['native_is_collision_validated'] is False
        assert path['all_segments_clear']
        rows.append([probe, path['component_id'], len(path['points']),
            f"{path['minimum_all_atom_clearance']:.9f}",
            f"{path['probe_margin']:.9f}", path['all_segments_clear'],
            path['native_is_collision_validated']])
table(['Probe (Å)', 'Component', 'Stations', 'Minimum clearance (Å)',
       'Probe margin (Å)', 'All interior segments clear', 'Native validated flag'], rows)

Both selected native interior polylines have minimum clearance **3.607542780 Å**,
so their 1.4 and 2.5 Å margins are positive. This is a useful positive result,
stronger than merely plotting a graph path. It covers those exact interior
segments. The independent axis reference separately supplies outward infinite
tails; the native skeleton does not. At 1.4 Å the Channel's physical-contact
records also retain their own checked attachment paths and exterior rays.

The axis reference has larger clearance, 3.906596981 Å. The native skeleton is
not optimized for global clearance, so that difference alone is not an error.
At 5 Å there is no multi-link candidate to inspect; absence of such a candidate
is not the proof of physical exclusion.

In [ ]:
path = np.asarray(candidates[2.5][0]['points'])
with study._tools():
    from devtools.verify_tube_passage import polyline_clearance
    per_segment = [polyline_clearance(path[i:i+2], centers, radii, length_unit='angstroms')
                   for i in range(len(path)-1)]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.9), layout='constrained')
axes[0].scatter(centers[:, 2], centers[:, 0], s=14, color='#507695', alpha=.6, label='Atomic centers')
axes[0].plot([-16, 16], [0, 0], ls='--', color='#007a58', label='Independent axis')
axes[0].plot(path[:, 2], path[:, 0], color='#be7718', lw=2, label='Native interior candidate, 2.5 Å')
axes[0].set(xlabel='z (Å)', ylabel='x (Å)', title='Projection of one interior candidate')
axes[0].legend(fontsize=8)
axes[1].plot(np.arange(len(per_segment)), per_segment, marker='.', color='#be7718', label='Exact all-atom segment minima')
axes[1].axhline(2.5, color='#ad3746', ls='--', label='Probe radius, 2.5 Å')
axes[1].axhline(references['passage']['path_minimum_clearance_angstroms'], color='#007a58', ls=':', label='Independent axis minimum')
axes[1].set(xlabel='Candidate segment index', ylabel='Clearance (Å)', title='Every segment remains clear')
axes[1].legend(fontsize=8); axes[1].grid(alpha=.2)
plt.show()

## Attribution, evidence and next decisions

The original capture and this new execution keep their **own producer versions**.
The original capture used committed scientific/helper source plus the retained
human version-file edit; its full source revision and version-file hash are in
the evidence JSON. The source-bound reader is checked separately against a clean
committed checkout. Neither route qualifies an installed release or platform matrix.

The first capture exposed a study-serialization defect: numeric-looking
bibliographic strings were parsed as quantity expressions. Its original attempt
is retained. The corrected capture verifies unchanged attribution metadata before
writing, under [#185](https://github.com/uibcdf/topomt/issues/185). This did not
change the scientific DFND operators or the three observed outcomes.

In [ ]:
rows = []
for probe, topo in results.items():
    attribution = topomt.get_attribution(topo)
    assert attribution['status'] == 'recorded'
    rows.append([probe, attribution['producer']['version'], attribution['status'],
                 len(attribution['references'])])
table(['Probe (Å)', 'Actual current producer version', 'Recording status', 'Reference records'], rows)
print('Original native source:', original['source_revision'])
print('Original producer:', original['probes'][0]['topography_attribution']['producer'])
# Citation rendering reads the completed result; it does not run another analysis.
bibliography = topomt.cite(results[1.4], format='bibtex')
assert bibliography and isinstance(bibliography, str)
print('BibTeX rendering succeeded; detached references and the application capture are available.')

The [paired tube summary](index.md#compare-the-two-tubes) shows the useful contrast:
at 2.5 Å the narrow hull is completely covered, while the wide input has a clear
route and independently verified end openings. Both geometry parameters change,
so this pair cannot attribute the difference to sampling alone.

Next work is explicit: numerical boundary handling [#184](https://github.com/uibcdf/topomt/issues/184),
bounded arrangements [#180](https://github.com/uibcdf/topomt/issues/180), local versus
all-atom/public qualification [#181](https://github.com/uibcdf/topomt/issues/181),
and wider certified paths [#76](https://github.com/uibcdf/topomt/issues/76).
The [finding register](https://github.com/uibcdf/topomt/discussions/154) owns their
current version-scoped interpretation. Extra local candidates in the positive
controls need support diagnosis within #181 too; two boundary patches do not
establish every three-dimensional component or public label.

No original third-party comparison was executed for either tube. Comparing
CAVER, MOLE or another method requires a separately frozen, eligible protocol
and matched definitions. No comparative superiority, optimal path, exact physical
volume, aperture area or biological accuracy claim follows from these results.